# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook provides an example for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
# Print some additional metadata fields
print(f"\nIdentifier: {metadata.identifier}\nPublished: {metadata.datePublished}\nVersion: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

A Croissant dataset is organized into one or more **RecordSets**. Each RecordSet can contain fields, which represent columns of tabular data or other entities.

Let's list all RecordSets and their associated fields and columns, referencing each entity by its `@id`.

In [ ]:
# List all available Record Sets
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}")
for rs in record_sets:
    print(f"\nRecord Set: {rs.id}")
    print(f"  Name: {rs.name if hasattr(rs, 'name') else rs.id}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else ''}")
    if hasattr(rs, 'fields') and rs.fields:
        print(f"  Fields:")
        for field in rs.fields:
            print(f"    - Field @id: {field.id}, name: {getattr(field, 'name', '')}")
        print()
    if hasattr(rs, 'columns') and rs.columns:
        print(f"  Columns:")
        for col in rs.columns:
            print(f"    - Column @id: {col.id}, name: {getattr(col, 'name', '')}")
        print()
if len(record_sets) == 0:
    print("No record sets found in this dataset metadata. The dataset may consist of only files or non-tabular data.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we extract all records from each available RecordSet (`@id`), using the `mlcroissant` API.

_**Note:** If the dataset includes multiple record sets, you can select one or more for further analysis._

In [ ]:
# If no record sets are defined in the metadata, fallback: try to access the default data table
# Otherwise, use the RecordSet ids found in the overview.
if len(record_sets) == 0:
    print("No structured record sets found. Please check the dataset or consult its documentation.")
    dataframes = {}
else:
    dataframes = {}
    # List of record set @ids
    record_set_ids = [rs.id for rs in record_sets]
    for record_set_id in record_set_ids:
        # Extract all records for a given record set
        print(f"\nLoading data for RecordSet @id: {record_set_id}")
        try:
            records = list(dataset.records(record_set=record_set_id))
            if records:
                dataframes[record_set_id] = pd.DataFrame(records)
                print(f"Loaded {len(records)} records.")
            else:
                print(f"No records found for RecordSet {record_set_id}.")
        except Exception as e:
            print(f"Could not load records for RecordSet {record_set_id}: {e}")

    # Pick the first non-empty dataframe for exploration:
    selected_record_set_id = None
    for rs_id, df in dataframes.items():
        if not df.empty:
            selected_record_set_id = rs_id
            break
    
    if selected_record_set_id:
        print(f"\nFields (columns) in RecordSet {selected_record_set_id}:\n", dataframes[selected_record_set_id].columns.tolist())
        display(dataframes[selected_record_set_id].head())
    else:
        print("No dataframes with records loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include operations such as removing outliers, transforming values, or grouping by key columns to prepare for further analysis.

_**Note:** Update the `numeric_field` and `group_field` variables below to match a numeric and categorical field (by column name or field `@id`) from the loaded data._

In [ ]:
# Example EDA: Filtering and normalizing one numeric field, and grouping by one categorical field.

import numpy as np

if 'selected_record_set_id' in locals() and selected_record_set_id and selected_record_set_id in dataframes:
    df = dataframes[selected_record_set_id]
    print(f"DataFrame for {selected_record_set_id} has {df.shape[0]} rows and {df.shape[1]} columns.")
    print(f"Columns: {df.columns.tolist()}")
    
    # Please inspect columns to select appropriate fields
    # For demonstration, let's pick the first numeric field (float or int) automatically
    numeric_field = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field = col
            break
    
    if numeric_field is None:
        print("No numeric field found in DataFrame for EDA.")
    else:
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
        # Filter records with numeric_field > threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"\nFiltered records where {numeric_field} > {threshold:.2f} (total {filtered_df.shape[0]} records):")
        display(filtered_df.head())
        # Normalize
        if filtered_df[numeric_field].std() != 0:
            filtered_df[f"{numeric_field}_normalized"] = (
                (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            )
            print(f"\nNormalized {numeric_field}:")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        else:
            print(f"Cannot normalize {numeric_field}; zero standard deviation.")

        # Attempt grouping by another field (categorical)
        group_field = None
        for col in df.columns:
            if col != numeric_field and (df[col].dtype == 'object' or df[col].dtype.name == 'category'):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example of visualizing the distribution of the selected numeric field, along with a boxplot grouped by a categorical field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and not filtered_df.empty and 'numeric_field' in locals() and numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field} (filtered)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to access the FAIR^2 dataset defined via a Croissant schema, discover its available record sets and fields (each referenced by their `@id`), extract data for analysis, and perform initial exploratory visualization and filtering using standard data science libraries.

Refer to the dataset's own documentation and schema definition for authoritative field mappings, advanced features, and deeper scientific analysis.

_Note: For further reproducible research, always reference entities using their `@id` to ensure consistency across dataset versions and schema changes._